In [4]:
import csv
import json
import random
from datetime import date, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260624
NUMBER_OF_CONTRACTS = 20_000

random.seed(RANDOM_SEED)


# ============================================================
# PROJECT PATHS
# ============================================================

# Jupyter is already running inside risk_scoring_project,
# so use its current folder directly.
PROJECT_ROOT = Path.cwd()

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "145", "168", "201", "268"
]

PRODUCTS = [
    {"product_code": "CZL3", "product_category": "CONSUMER_LOAN"},
    {"product_code": "AUTO", "product_category": "AUTO_LOAN"},
    {"product_code": "MORT", "product_category": "MORTGAGE"},
    {"product_code": "BIZL", "product_category": "SME_LOAN"},
    {"product_code": "PERS", "product_category": "PERSONAL_LOAN"},
    {"product_code": "CRED", "product_category": "CREDIT_LINE"},
]

CUSTOMER_IDS = [str(customer_id) for customer_id in range(4_000_001, 4_015_001)]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    """Returns one random date between two dates."""

    number_of_days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, number_of_days))


def generate_account_number(
    branch_code: str,
    product_code: str,
    book_date: date,
    sequence_number: int
) -> str:
    """
    Creates a 16-character account number.

    Example:
    268CZL3171380001
    """

    year_part = book_date.strftime("%y")
    sequence_part = f"{sequence_number:07d}"

    return f"{branch_code}{product_code}{year_part}{sequence_part}"


def generate_application_number(
    book_date: date,
    sequence_number: int
) -> str:
    """
    Generates a maximum 16-character application number.
    Example: APP2600000000001
    """

    return f"APP{book_date:%y}{sequence_number:011d}"


def generate_contract_row(sequence_number: int) -> dict:
    """
    Generates one clean contract-level record.

    These fields are the identity and relationship fields needed
    before we generate the remaining loan attributes.
    """

    branch_code = random.choice(BRANCHES)
    product = random.choice(PRODUCTS)

    book_date = random_date(
        start_date=date(2018, 1, 1),
        end_date=date(2026, 5, 31)
    )

    account_number = generate_account_number(
        branch_code=branch_code,
        product_code=product["product_code"],
        book_date=book_date,
        sequence_number=sequence_number
    )

    customer_id = random.choice(CUSTOMER_IDS)

    return {
        "ACCOUNT_NUMBER": account_number,
        "BRANCH_CODE": branch_code,
        "APPLICATION_NUM": generate_application_number(
            book_date=book_date,
            sequence_number=sequence_number
        ),
        "CUSTOMER_ID": customer_id,
        "PRODUCT_CODE": product["product_code"],
        "PRODUCT_CATEGORY": product["product_category"],
        "BOOK_DATE": book_date.isoformat(),
        "VALUE_DATE": book_date.isoformat(),
        "ORIGINAL_ST_DATE": book_date.isoformat(),
        "PRIMARY_APPLICANT_ID": customer_id,
        "ALT_ACC_NO": account_number
    }


def validate_contracts(contract_rows: list[dict]) -> None:
    """
    Stops the script if contract identifiers are invalid.
    """

    account_numbers = [
        row["ACCOUNT_NUMBER"]
        for row in contract_rows
    ]

    application_numbers = [
        row["APPLICATION_NUM"]
        for row in contract_rows
    ]

    if len(contract_rows) != NUMBER_OF_CONTRACTS:
        raise ValueError(
            f"Expected {NUMBER_OF_CONTRACTS:,} rows, "
            f"but generated {len(contract_rows):,}."
        )

    if len(account_numbers) != len(set(account_numbers)):
        raise ValueError("Duplicate ACCOUNT_NUMBER values were generated.")

    if len(application_numbers) != len(set(application_numbers)):
        raise ValueError("Duplicate APPLICATION_NUM values were generated.")

    if any(not account_number for account_number in account_numbers):
        raise ValueError("A blank ACCOUNT_NUMBER was generated.") 

    if any(len(account_number) > 35 for account_number in account_numbers):
        raise ValueError(
            "At least one ACCOUNT_NUMBER exceeds VARCHAR2(35)."
        )


def write_csv(file_path: Path, contract_rows: list[dict]) -> None:
    """Writes contract data to CSV."""

    column_names = list(contract_rows[0].keys())

    with open(file_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=column_names
        )

        writer.writeheader()
        writer.writerows(contract_rows)


# ============================================================
# MAIN SCRIPT
# ============================================================

def main() -> None:
    print(f"Generating {NUMBER_OF_CONTRACTS:,} unique contracts...")

    contract_rows = [
        generate_contract_row(sequence_number=sequence_number)
        for sequence_number in range(1, NUMBER_OF_CONTRACTS + 1)
    ]

    validate_contracts(contract_rows)

    full_output_path = RAW_DATA_FOLDER / "rs_contracts_base.csv"
    sample_output_path = SAMPLE_DATA_FOLDER / "rs_contracts_base_sample.csv"
    summary_output_path = RAW_DATA_FOLDER / "rs_contracts_generation_summary.json"

    write_csv(
        file_path=full_output_path,
        contract_rows=contract_rows
    )

    write_csv(
        file_path=sample_output_path,
        contract_rows=contract_rows[:100]
    )

    generation_summary = {
        "table_name": "RS_CONTRACTS",
        "generated_rows": len(contract_rows),
        "unique_account_numbers": len(
            {row["ACCOUNT_NUMBER"] for row in contract_rows}
        ),
        "unique_application_numbers": len(
            {row["APPLICATION_NUM"] for row in contract_rows}
        ),
        "customer_id_pool_size": len(CUSTOMER_IDS),
        "random_seed": RANDOM_SEED,
        "full_file": str(full_output_path),
        "sample_file": str(sample_output_path)
    }

    with open(summary_output_path, "w", encoding="utf-8") as file:
        json.dump(
            generation_summary,
            file,
            indent=4
        )

    print("Generation completed successfully.")
    print(f"Full dataset: {full_output_path}")
    print(f"Sample dataset: {sample_output_path}")
    print(f"Unique ACCOUNT_NUMBER count: {generation_summary['unique_account_numbers']:,}")


if __name__ == "__main__":
    main()

Generating 20,000 unique contracts...
Generation completed successfully.
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_contracts_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_contracts_base_sample.csv
Unique ACCOUNT_NUMBER count: 20,000


In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\ASUS\Desktop\project\risk_scoring_project


In [5]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260625
NUMBER_OF_CUSTOMERS = 15_000

# This matches the customer ID pool used in generate_contracts.py.
CUSTOMER_ID_START = 4_000_001
CUSTOMER_ID_END = 4_015_000

random.seed(RANDOM_SEED)


# ============================================================
# NOTEBOOK-SAFE PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CONTRACT_FILE = PROJECT_ROOT / "data" / "raw" / "rs_contracts_base.csv"
RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# CUSTOMER TABLE COLUMNS
# ============================================================

CUSTOMER_COLUMNS = [
    "GEN_DIM_CUST_ID",
    "CUSTOMER_NO",
    "PIN",
    "DOCUMENT_NO",
    "TAX_ID",
    "CUSTOMER_TYPE",
    "CUSTOMER_SUB_TYPE",
    "FULL_NAME",
    "FIRST_NAME",
    "LAST_NAME",
    "MIDDLE_NAME",
    "GENDER",
    "BIRTH_DT",
    "PLACE_OF_BIRTH",
    "MARITAL_STATUS",
    "NATIONALITY",
    "OPEN_REASON",
    "CREATION_DT",
    "OPENING_BRN",
    "FLG_IS_COURT_ISSUE",
    "OCCUPATION",
    "WORKPLACE_CUSTOMER_NO",
    "WORKPLACE_NAME",
    "FLG_IS_BLACK_LIST",
    "FLG_IS_VIP",
    "FLG_IS_EMPLOYEER",
    "NUM_OF_EMPLOYEER",
    "RM_NAME",
    "CUSTOMER_GENERATION",
    "FLG_IS_AFFLUENT",
    "FLG_IS_SALARY",
    "FLG_IS_PENSION",
    "FLG_IS_EDV",
    "RISK_LEVEL",
    "RESIDENT_STATUS",
    "RM_ID_CORP",
    "GROUP_ID_CORP",
    "CHECKER_DT_STAMP",
    "MAKER_DT_STAMP",
    "EMPL_PROPERTY_TYPE",
    "SHORT_NAME",
    "COUNTRY",
    "DOCUMENT_NAME",
    "LEGAL_GUARDIAN",
    "PASSPORT_COUNTRY",
    "PASSPORT_NATIONAL_ID",
    "PASSPORT_ISS_DT",
    "PASSPORT_EXP_DT",
    "P_ADDRESS1",
    "P_ADDRESS2",
    "P_ADDRESS3",
    "ENGLISH_FULL_NAME",
    "FLG_IS_BENEFICIARY",
    "CUSTOMER_SINCE",
    "DIRECTOR_CUST_NO",
    "FLG_IS_GREEN_CARD",
    "US_BIRTH",
    "FLG_IS_US_CITIZEN",
    "US_PAYMENT",
    "FLG_IS_US_RESIDENT_POA",
    "FLG_IS_US_ADDRESS",
    "FLG_IS_US_LIVING",
    "US_PASSPORT",
    "FLG_IS_US_PHONE",
    "FLG_IS_US_POA",
    "FLG_IS_US_POST_BOX",
    "US_RELATED_PERSON",
    "FLG_IS_US_TAX_RESIDENT",
    "WORKPLACE_ADDRESS",
    "WORKPLACE",
    "CIF_RECORD_STATUS",
    "FLG_IS_FROZEN",
    "INS_DTTM",
    "TAX_TYPE",
    "FLG_IS_INGROUP",
    "CUST_MIS_1",
    "SHORT_NAME2",
    "SWIFT_CD",
    "CHARGE_GROUP",
    "DEFAULT_MEDIA",
    "CUSTOMER_ADDRESS",
    "CORP_REGISTER_ADDRESS",
    "CORP_REGISTER_COUNTRY",
    "CHECKER_DT",
    "FLG_IS_MILITARY_PASSPORT",
    "CUST_SUB_SEGMENT",
    "AGRAR_PKD",
    "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY",
    "FLG_IS_REMOTE_ACC_ALLOWED",
    "ONLY_LAST_AND_FIRST_NAME_ENG",
    "EMPLOYER_ID",
    "INCORP_DATE",
    "DT_QHT_GROUP",
    "CUST_CLASSIFICATION",
    "LOAN_RESP_ID",
    "ULTIMATE_BENEFICIAL_OWNER",
    "FLG_IS_RELATED_PERSON",
    "COUNTRY_DESC",
    "REGISTRATION_ADDRESS",
    "DOMICILE_ADDRESS",
]


# ============================================================
# REFERENCE DATA
# ============================================================

BRANCHES = [
    "001", "007", "015", "022", "031",
    "042", "056", "071", "089", "102",
    "118", "120", "145", "168", "201", "268"
]

CITIES = [
    "BAKI", "SUMQAYIT", "GANCA", "MINGECEVIR", "SIRVAN",
    "LENKERAN", "QUBA", "SEKI", "MASALLI", "SAMAXI",
    "YEVLAX", "QABALA", "NAXCIVAN", "ZAGATALA"
]

STREETS = [
    "NIZAMI KUCHESI",
    "ATATURK PROSPEKTI",
    "HEYDER ELIYEV PROSPEKTI",
    "XETAI PROSPEKTI",
    "TBLISI PROSPEKTI",
    "BABEK PROSPEKTI",
    "SULEYMAN RUSTEM KUCHESI",
    "AZADLIQ PROSPEKTI"
]

MALE_FIRST_NAMES = [
    "Elvin", "Rashad", "Murad", "Tural", "Orkhan", "Kamran",
    "Nihad", "Samir", "Fuad", "Javid", "Anar", "Emil",
    "Rovshan", "Elnur", "Aydin", "Teymur", "Ramil", "Aqil"
]

FEMALE_FIRST_NAMES = [
    "Aysel", "Nigar", "Leyla", "Gunel", "Sevda", "Narmin",
    "Aynur", "Khadija", "Sabina", "Aytac", "Zahra", "Arzu",
    "Laman", "Ulker", "Fatima", "Rena", "Aysu", "Turan"
]

FATHER_NAMES = [
    "Rasim", "Vugar", "Camil", "Tahir", "Nazim", "Farhad",
    "Ilham", "Elman", "Adil", "Rauf", "Natiq", "Samad"
]

SURNAMES = [
    "Mammadov", "Aliyev", "Hasanov", "Huseynov", "Ibrahimov",
    "Rahimov", "Karimov", "Quliyev", "Babayeva", "Hajiyeva",
    "Asgarov", "Jafarov", "Suleymanov", "Nabiyev", "Ismayilov",
    "Abbasov", "Rzayev", "Safarov", "Tagiyev", "Aghayev"
]

OCCUPATIONS = [
    "ACCOUNTANT", "TEACHER", "ENGINEER", "DOCTOR",
    "SALES SPECIALIST", "IT SPECIALIST", "DRIVER",
    "MANAGER", "LAWYER", "ANALYST", "ENTREPRENEUR",
    "TECHNICIAN", "CIVIL SERVANT", "DESIGNER"
]

WORKPLACES = [
    "CASPIAN LOGISTICS MMC",
    "BAKU RETAIL MMC",
    "AZER TELECOM MMC",
    "NORTH CONSTRUCTION MMC",
    "CITY EDUCATION CENTER",
    "MEDICAL SERVICE MMC",
    "FINANCE CONSULTING MMC",
    "REGIONAL TRADE MMC",
    "TECH SOLUTIONS MMC"
]

COMPANY_PREFIXES = [
    "Caspian", "Araz", "Baku", "Shirvan", "Absheron",
    "Zirve", "Khazar", "North", "Silk Road", "Azer",
    "Green", "Atlas"
]

COMPANY_ACTIVITIES = [
    "Logistics", "Construction", "Trade", "Food",
    "Technology", "Textile", "Agriculture", "Pharmacy",
    "Tourism", "Energy", "Consulting", "Manufacturing",
    "Transport", "Services"
]

COMPANY_SUFFIXES = ["MMC", "ASC"]

RELATIONSHIP_MANAGERS = [
    "A. Mammadov",
    "L. Aliyeva",
    "R. Hasanov",
    "N. Ibrahimova",
    "K. Huseynov",
    "S. Rahimova"
]

SWIFT_CODES = [
    "AZEBAZ22",
    "BAKUAJ22",
    "CASPAZ22",
    "FINAAZ22"
]

PIN_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ0123456789"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def random_date(start_date: date, end_date: date) -> date:
    if end_date < start_date:
        return start_date

    days_between = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, days_between))


def format_date(value):
    return value.isoformat() if value else ""


def format_timestamp(value):
    return value.strftime("%Y-%m-%d %H:%M:%S") if value else ""


def years_before(reference_date: date, years: int) -> date:
    try:
        return reference_date.replace(year=reference_date.year - years)
    except ValueError:
        return reference_date.replace(
            year=reference_date.year - years,
            month=2,
            day=28
        )


def generation_label(birth_date: date) -> str:
    if birth_date.year <= 1964:
        return "BABY_BOOMERS"
    if birth_date.year <= 1980:
        return "GEN_X"
    if birth_date.year <= 1996:
        return "MILLENNIALS_GEN_Y"

    return "GEN_Z"


def parse_iso_date(value: str):
    if not value:
        return None

    try:
        return datetime.strptime(value.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


def generate_unique_pin(existing_pins: set) -> str:
    while True:
        pin = (
            str(random.randint(1, 9))
            + "".join(random.choices(PIN_ALPHABET, k=6))
        )

        if pin not in existing_pins:
            existing_pins.add(pin)
            return pin


def generate_unique_document_number(existing_documents: set) -> str:
    while True:
        document_number = f"AZE{random.randint(10_000_000, 99_999_999)}"

        if document_number not in existing_documents:
            existing_documents.add(document_number)
            return document_number


def generate_unique_tax_id(existing_tax_ids: set) -> str:
    while True:
        tax_id = str(random.randint(1_000_000_000, 9_999_999_999))

        if tax_id not in existing_tax_ids:
            existing_tax_ids.add(tax_id)
            return tax_id


def make_address(city: str) -> str:
    return (
        f"{city}, {random.choice(STREETS)}, "
        f"EV {random.randint(1, 250)}, MEN {random.randint(1, 150)}"
    )


# ============================================================
# CONTRACT READING AND KEY MATCHING
# ============================================================

def load_contract_customer_details():
    """
    Reads contracts and collects:
    - customer IDs
    - earliest contract date per customer
    - products per customer
    """

    if not CONTRACT_FILE.exists():
        raise FileNotFoundError(
            f"Contract file not found: {CONTRACT_FILE}\n"
            "First run the contract generator."
        )

    customer_ids = set()
    first_contract_date = {}
    product_codes = defaultdict(set)

    with open(
        CONTRACT_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_ID",
            "BOOK_DATE",
            "PRODUCT_CODE"
        }

        missing_columns = required_columns - set(reader.fieldnames or [])

        if missing_columns:
            raise ValueError(
                "Missing contract columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_id = (row.get("CUSTOMER_ID") or "").strip()

            if not customer_id:
                continue

            customer_ids.add(customer_id)

            book_date = parse_iso_date(
                row.get("BOOK_DATE") or ""
            )

            if book_date:
                current_minimum = first_contract_date.get(customer_id)

                if (
                    current_minimum is None
                    or book_date < current_minimum
                ):
                    first_contract_date[customer_id] = book_date

            product_code = (row.get("PRODUCT_CODE") or "").strip()

            if product_code:
                product_codes[customer_id].add(product_code)

    return customer_ids, first_contract_date, product_codes


def build_customer_numbers(contract_customer_ids: set) -> list:
    """
    Creates exactly 15,000 CIF values.

    Every CUSTOMER_ID in contracts is included here, guaranteeing:
    CUSTOMER_ID = CUSTOMER_NO.
    """

    customer_numbers = [
        str(customer_id)
        for customer_id in range(
            CUSTOMER_ID_START,
            CUSTOMER_ID_END + 1
        )
    ]

    if len(customer_numbers) != NUMBER_OF_CUSTOMERS:
        raise ValueError(
            "Customer ID range must create exactly "
            f"{NUMBER_OF_CUSTOMERS:,} records."
        )

    unmatched_contract_ids = (
        contract_customer_ids - set(customer_numbers)
    )

    if unmatched_contract_ids:
        examples = ", ".join(sorted(unmatched_contract_ids)[:5])

        raise ValueError(
            "Some contract CUSTOMER_ID values are outside the expected "
            f"customer-ID range. Examples: {examples}"
        )

    random.shuffle(customer_numbers)

    return customer_numbers


def choose_customer_type(product_codes: set) -> str:
    """
    Business loans produce more corporate customers.
    Consumer loans produce mostly individual customers.
    """

    if "BIZL" in product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[35, 63, 2],
            k=1
        )[0]

    if product_codes:
        return random.choices(
            ["I", "C", "B"],
            weights=[88, 11, 1],
            k=1
        )[0]

    return random.choices(
        ["I", "C", "B"],
        weights=[80, 19, 1],
        k=1
    )[0]


# ============================================================
# ROW GENERATORS
# ============================================================

def make_base_row(customer_no: str, generated_id: int) -> dict:
    row = {
        column_name: ""
        for column_name in CUSTOMER_COLUMNS
    }

    row.update({
        "GEN_DIM_CUST_ID": str(generated_id),
        "CUSTOMER_NO": customer_no,
        "FLG_IS_COURT_ISSUE": "N",
        "FLG_IS_BLACK_LIST": "N",
        "FLG_IS_VIP": "N",
        "FLG_IS_EMPLOYEER": "N",
        "FLG_IS_EDV": "Y",
        "FLG_IS_BENEFICIARY": "N",
        "FLG_IS_GREEN_CARD": "N",
        "US_BIRTH": "N",
        "FLG_IS_US_CITIZEN": "N",
        "US_PAYMENT": "N",
        "FLG_IS_US_RESIDENT_POA": "N",
        "FLG_IS_US_ADDRESS": "N",
        "FLG_IS_US_LIVING": "N",
        "US_PASSPORT": "N",
        "FLG_IS_US_PHONE": "N",
        "FLG_IS_US_POA": "N",
        "FLG_IS_US_POST_BOX": "N",
        "US_RELATED_PERSON": "N",
        "FLG_IS_US_TAX_RESIDENT": "N",
        "CIF_RECORD_STATUS": "O",
        "FLG_IS_FROZEN": "N",
        "FLG_IS_INGROUP": "N",
        "FLG_IS_MILITARY_PASSPORT": "N",
        "FLG_IS_ACTIVE_CUSTOMER_QUARTERLY": "Y",
        "FLG_IS_REMOTE_ACC_ALLOWED": "Y",
        "FLG_IS_RELATED_PERSON": "N",
        "COUNTRY": "AZ",
        "COUNTRY_DESC": "AZERBAIJAN",
        "PASSPORT_COUNTRY": "AZ",
        "RESIDENT_STATUS": "R",
        "RISK_LEVEL": str(
            random.choices(
                [1, 2, 3, 4, 5],
                weights=[10, 30, 35, 20, 5],
                k=1
            )[0]
        ),
        "DEFAULT_MEDIA": random.choice(
            ["SMS", "EMAIL", "MOBILE_APP"]
        ),
        "OPENING_BRN": random.choice(BRANCHES),
        "INS_DTTM": "2026-06-21",
        "CHECKER_DT": "2026-06-21",
    })

    return row


def generate_individual(
    row: dict,
    first_contract_date: date,
    existing_pins: set,
    existing_documents: set
) -> dict:
    gender = random.choice(["M", "F"])

    first_name = (
        random.choice(MALE_FIRST_NAMES)
        if gender == "M"
        else random.choice(FEMALE_FIRST_NAMES)
    )

    last_name = random.choice(SURNAMES)
    father_name = random.choice(FATHER_NAMES)

    middle_name = (
        f"{father_name} OGLU"
        if gender == "M"
        else f"{father_name} QIZI"
    )

    age_at_loan = random.randint(21, 72)

    birth_date = years_before(
        first_contract_date,
        age_at_loan
    )

    birth_date -= timedelta(days=random.randint(0, 364))

    customer_creation_start = max(
        date(2005, 1, 1),
        birth_date + timedelta(days=18 * 365)
    )

    creation_date = random_date(
        customer_creation_start,
        first_contract_date
    )

    passport_issue_date = random_date(
        max(
            birth_date + timedelta(days=16 * 365),
            date(2010, 1, 1)
        ),
        creation_date
    )

    passport_expiry_date = (
        passport_issue_date
        + timedelta(days=random.randint(7 * 365, 10 * 365))
    )

    city = random.choice(CITIES)
    occupation = random.choice(OCCUPATIONS)
    workplace = random.choice(WORKPLACES)

    is_salary_customer = random.choices(
        ["Y", "N"],
        weights=[58, 42],
        k=1
    )[0]

    is_pensioner = (
        "Y"
        if age_at_loan >= 63 and random.random() < 0.55
        else "N"
    )

    is_affluent = "Y" if random.random() < 0.08 else "N"

    is_vip = (
        "Y"
        if is_affluent == "Y" and random.random() < 0.12
        else "N"
    )

    is_employee = "Y" if random.random() < 0.02 else "N"

    full_name = f"{last_name} {first_name} {middle_name}"
    english_name = f"{last_name.upper()} {first_name.upper()}"

    row.update({
        "PIN": generate_unique_pin(existing_pins),
        "DOCUMENT_NO": generate_unique_document_number(
            existing_documents
        ),
        "CUSTOMER_TYPE": "I",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["INDIVIDUAL", "RETAIL", "INDIVIDUAL OWNER"],
            weights=[50, 42, 8],
            k=1
        )[0],
        "FULL_NAME": full_name,
        "FIRST_NAME": first_name,
        "LAST_NAME": last_name,
        "MIDDLE_NAME": middle_name,
        "GENDER": gender,
        "BIRTH_DT": format_date(birth_date),
        "PLACE_OF_BIRTH": city,
        "MARITAL_STATUS": random.choices(
            ["S", "M", "D", "W"],
            weights=[35, 54, 7, 4],
            k=1
        )[0],
        "NATIONALITY": "AZ",
        "OPEN_REASON": (
            "PENSION"
            if is_pensioner == "Y"
            else "SALARY"
            if is_salary_customer == "Y"
            else random.choice(["CARD", "ABBM", "LOAN"])
        ),
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": occupation,
        "WORKPLACE_CUSTOMER_NO": (
            str(random.randint(5_000_001, 5_050_000))
            if random.random() < 0.45
            else ""
        ),
        "WORKPLACE_NAME": workplace,
        "FLG_IS_VIP": is_vip,
        "FLG_IS_EMPLOYEER": is_employee,
        "RM_NAME": (
            random.choice(RELATIONSHIP_MANAGERS)
            if is_affluent == "Y"
            else ""
        ),
        "CUSTOMER_GENERATION": generation_label(birth_date),
        "FLG_IS_AFFLUENT": is_affluent,
        "FLG_IS_SALARY": is_salary_customer,
        "FLG_IS_PENSION": is_pensioner,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=12,
                minutes=random.randint(0, 59)
            )
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(
                hours=11,
                minutes=random.randint(0, 59)
            )
        ),
        "EMPL_PROPERTY_TYPE": random.choice([
            "PRIVATE_SECTOR",
            "PUBLIC_SECTOR",
            "SELF_EMPLOYED"
        ]),
        "SHORT_NAME": (
            f"{first_name[:7].upper()}{row['PIN'][:7]}"
        )[:20],
        "DOCUMENT_NAME": "IDENTITY_CARD",
        "PASSPORT_NATIONAL_ID": row["PIN"],
        "PASSPORT_ISS_DT": format_date(passport_issue_date),
        "PASSPORT_EXP_DT": format_date(passport_expiry_date),
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": english_name,
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(
            random.choice(CITIES)
        ),
        "WORKPLACE": workplace,
        "CUST_MIS_1": "RETAIL",
        "SHORT_NAME2": (
            f"{last_name[:8].upper()}{first_name[:4].upper()}"
        )[:20],
        "CHARGE_GROUP": "FIZIKI",
        "CUSTOMER_ADDRESS": make_address(city),
        "CUST_SUB_SEGMENT": (
            "AFFLUENT"
            if is_affluent == "Y"
            else "MASS"
        ),
        "ONLY_LAST_AND_FIRST_NAME_ENG": english_name,
        "EMPLOYER_ID": (
            f"EMP{random.randint(1000, 9999)}"
            if is_employee == "Y"
            else ""
        ),
        "CUST_CLASSIFICATION": (
            "VIP"
            if is_vip == "Y"
            else "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
    })

    return row


def generate_company(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    company_sequence: int
) -> dict:
    company_name = (
        f"{random.choice(COMPANY_PREFIXES)} "
        f"{random.choice(COMPANY_ACTIVITIES)} "
        f"{random.choice(COMPANY_SUFFIXES)} "
        f"{company_sequence:04d}"
    )

    incorporation_date = random_date(
        date(1995, 1, 1),
        first_contract_date - timedelta(days=30)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = random.choice(CITIES)

    employee_count = random.choices(
        [
            random.randint(5, 49),
            random.randint(50, 249),
            random.randint(250, 3000)
        ],
        weights=[60, 32, 8],
        k=1
    )[0]

    is_affluent = (
        "Y"
        if employee_count >= 250 or random.random() < 0.18
        else "N"
    )

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "C",
        "CUSTOMER_SUB_TYPE": random.choices(
            ["SME/C", "CORPORATE"],
            weights=[72, 28],
            k=1
        )[0],
        "FULL_NAME": company_name,
        "OPEN_REASON": "BUSINESS",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "LEGAL_ENTITY",
        "NUM_OF_EMPLOYEER": str(employee_count),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": is_affluent,
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "LEGAL_ENTITY",
        "SHORT_NAME": company_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "TAX_CERTIFICATE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": company_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": company_name,
        "TAX_TYPE": "LEGAL_ENTITY",
        "CUST_MIS_1": "CORP",
        "SHORT_NAME2": company_name[:20].upper(),
        "CHARGE_GROUP": "KORP",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": (
            "LARGE_CORP"
            if employee_count >= 250
            else "SME"
        ),
        "INCORP_DATE": format_date(incorporation_date),
        "DT_QHT_GROUP": f"GRP_{random.randint(1, 300):04d}",
        "CUST_CLASSIFICATION": (
            "PREMIUM"
            if is_affluent == "Y"
            else "STANDARD"
        ),
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "FLG_IS_RELATED_PERSON": (
            "Y"
            if random.random() < 0.03
            else "N"
        ),
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"GRP{random.randint(1, 300):04d}",
    })

    return row


def generate_bank(
    row: dict,
    first_contract_date: date,
    existing_tax_ids: set,
    bank_sequence: int
) -> dict:
    bank_name = (
        f"{random.choice(['Caspian', 'Azer', 'Shirvan', 'Atlas'])} "
        f"Bank ASC {bank_sequence:03d}"
    )

    incorporation_date = random_date(
        date(1992, 1, 1),
        first_contract_date - timedelta(days=365)
    )

    creation_date = random_date(
        incorporation_date,
        first_contract_date
    )

    city = "BAKI"

    row.update({
        "TAX_ID": generate_unique_tax_id(existing_tax_ids),
        "CUSTOMER_TYPE": "B",
        "CUSTOMER_SUB_TYPE": "BANK",
        "FULL_NAME": bank_name,
        "OPEN_REASON": "CORRESPONDENT",
        "CREATION_DT": format_date(creation_date),
        "OCCUPATION": "BANK",
        "NUM_OF_EMPLOYEER": str(
            random.randint(200, 2500)
        ),
        "RM_NAME": random.choice(RELATIONSHIP_MANAGERS),
        "FLG_IS_AFFLUENT": "Y",
        "CHECKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=12)
        ),
        "MAKER_DT_STAMP": format_timestamp(
            datetime.combine(
                creation_date,
                datetime.min.time()
            ) + timedelta(hours=11)
        ),
        "EMPL_PROPERTY_TYPE": "BANK",
        "SHORT_NAME": bank_name.replace(" ", "")[:20].upper(),
        "DOCUMENT_NAME": "BANK_LICENSE",
        "P_ADDRESS1": make_address(city),
        "ENGLISH_FULL_NAME": bank_name.upper(),
        "CUSTOMER_SINCE": creation_date.strftime("%Y-%m"),
        "WORKPLACE_ADDRESS": make_address(city),
        "WORKPLACE": bank_name,
        "TAX_TYPE": "FINANCIAL_INSTITUTION",
        "CUST_MIS_1": "BANK",
        "SHORT_NAME2": bank_name[:20].upper(),
        "SWIFT_CD": random.choice(SWIFT_CODES),
        "CHARGE_GROUP": "BANK",
        "CUSTOMER_ADDRESS": make_address(city),
        "CORP_REGISTER_ADDRESS": make_address(city),
        "CORP_REGISTER_COUNTRY": "AZ",
        "CUST_SUB_SEGMENT": "BANK",
        "INCORP_DATE": format_date(incorporation_date),
        "CUST_CLASSIFICATION": "PREMIUM",
        "LOAN_RESP_ID": f"LR{random.randint(1000, 9999)}",
        "RM_ID_CORP": f"RM{random.randint(1001, 1010)}",
        "GROUP_ID_CORP": f"BNK{bank_sequence:03d}",
    })

    return row


# ============================================================
# VALIDATION
# ============================================================

def validate_customer_rows(
    customer_rows: list,
    contract_customer_ids: set
) -> None:
    customer_numbers = [
        row["CUSTOMER_NO"]
        for row in customer_rows
    ]

    individual_pins = [
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] == "I"
    ]

    if len(customer_rows) != NUMBER_OF_CUSTOMERS:
        raise ValueError("Customer row count is incorrect.")

    if len(customer_numbers) != len(set(customer_numbers)):
        raise ValueError("Duplicate CUSTOMER_NO / CIF values exist.")

    if any(not customer_no for customer_no in customer_numbers):
        raise ValueError("Blank CUSTOMER_NO / CIF values exist.")

    if len(individual_pins) != len(set(individual_pins)):
        raise ValueError("Duplicate individual PIN values exist.")

    if any(not pin for pin in individual_pins):
        raise ValueError("An individual customer has no PIN.")

    if any(
        row["PIN"]
        for row in customer_rows
        if row["CUSTOMER_TYPE"] in {"C", "B"}
    ):
        raise ValueError(
            "A company or bank incorrectly has a PIN."
        )

    missing_customers = (
        contract_customer_ids - set(customer_numbers)
    )

    if missing_customers:
        raise ValueError(
            "Some contract CUSTOMER_ID values have no "
            "matching CUSTOMER_NO."
        )


def write_csv(file_path: Path, rows: list) -> None:
    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=CUSTOMER_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main():
    (
        contract_customer_ids,
        first_contract_date_by_customer,
        product_codes_by_customer
    ) = load_contract_customer_details()

    customer_numbers = build_customer_numbers(
        contract_customer_ids
    )

    existing_pins = set()
    existing_documents = set()
    existing_tax_ids = set()

    customer_rows = []

    individual_count = 0
    corporate_count = 0
    bank_count = 0

    for row_number, customer_no in enumerate(
        customer_numbers,
        start=1
    ):
        first_contract_date = first_contract_date_by_customer.get(
            customer_no,
            date(2026, 5, 31)
        )

        customer_products = product_codes_by_customer.get(
            customer_no,
            set()
        )

        customer_type = choose_customer_type(customer_products)

        row = make_base_row(
            customer_no=customer_no,
            generated_id=100_000_000 + row_number
        )

        if customer_type == "I":
            individual_count += 1

            row = generate_individual(
                row=row,
                first_contract_date=first_contract_date,
                existing_pins=existing_pins,
                existing_documents=existing_documents
            )

        elif customer_type == "C":
            corporate_count += 1

            row = generate_company(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                company_sequence=corporate_count
            )

        else:
            bank_count += 1

            row = generate_bank(
                row=row,
                first_contract_date=first_contract_date,
                existing_tax_ids=existing_tax_ids,
                bank_sequence=bank_count
            )

        customer_rows.append(row)

    validate_customer_rows(
        customer_rows=customer_rows,
        contract_customer_ids=contract_customer_ids
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_customers_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER / "rs_customers_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER / "rs_customers_generation_summary.json"
    )

    write_csv(full_output_file, customer_rows)
    write_csv(sample_output_file, customer_rows[:100])

    summary = {
        "table_name": "RS_CUSTOMERS",
        "generated_rows": len(customer_rows),
        "contract_customer_ids_found": len(contract_customer_ids),
        "individual_customers": individual_count,
        "corporate_customers": corporate_count,
        "bank_customers": bank_count,
        "unique_customer_numbers": len(
            {row["CUSTOMER_NO"] for row in customer_rows}
        ),
        "unique_individual_pins": len(existing_pins),
        "contract_customer_ids_without_match": 0,
        "join_rule": (
            "RS_CONTRACTS.CUSTOMER_ID "
            "= RS_CUSTOMERS.CUSTOMER_NO"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("Customer generation completed successfully.")
    print(f"Generated customers: {len(customer_rows):,}")
    print(f"Individuals: {individual_count:,}")
    print(f"Corporate customers: {corporate_count:,}")
    print(f"Banks: {bank_count:,}")
    print(
        "Contract-to-customer unmatched IDs: 0"
    )
    print(f"Full dataset: {full_output_file}")
    print(f"Sample dataset: {sample_output_file}")


if __name__ == "__main__":
    main()

Customer generation completed successfully.
Generated customers: 15,000
Individuals: 11,363
Corporate customers: 3,482
Banks: 155
Contract-to-customer unmatched IDs: 0
Full dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_customers_base.csv
Sample dataset: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_customers_base_sample.csv


In [1]:
import csv
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260626
NUMBER_OF_MKR_RECORDS = 40_000

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

# This works inside your Jupyter notebook.
PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# MKR MASTER TABLE COLUMNS
# ============================================================

MKR_MASTER_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "BALANCE",
    "REPORTID",
    "REPORTINGDATE",
    "INSERT_DATE",
    "DOCUMENT_NO"
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(value.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between start_date and end_date."""

    if start_date >= end_date:
        return start_date

    day_difference = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, day_difference)
    )


def get_customer_type(customer_row):
    """
    I = individual
    C = company
    B = bank

    If CUSTOMER_TYPE is unexpectedly blank,
    PIN presence is used as a fallback rule.
    """

    customer_type = (
        customer_row.get("CUSTOMER_TYPE") or ""
    ).strip().upper()

    if customer_type in {"I", "C", "B"}:
        return customer_type

    if (customer_row.get("PIN") or "").strip():
        return "I"

    return "C"


def generate_balance(customer_type):
    """
    Creates a realistic total reported liability balance.

    Later, when we create the liabilities child table,
    liability balances will sum exactly to this value.
    """

    if customer_type == "I":
        # Some people may have no active liability on the inquiry date.
        if random.random() < 0.18:
            return 0.00

        return round(
            random.triangular(
                low=200,
                high=30_000,
                mode=4_000
            ),
            2
        )

    if customer_type == "C":
        # Companies normally carry higher balances.
        if random.random() < 0.08:
            return 0.00

        return round(
            random.triangular(
                low=5_000,
                high=2_000_000,
                mode=180_000
            ),
            2
        )

    # Banks / financial institutions.
    if random.random() < 0.05:
        return 0.00

    return round(
        random.triangular(
            low=500_000,
            high=8_000_000,
            mode=2_000_000
        ),
        2
    )


def load_customers():
    """
    Reads the customer master dataset and returns:
    - customers_by_cif: dictionary where CIF is the key
    - customer_rows: all customer records
    """

    if not CUSTOMER_FILE.exists():
        raise FileNotFoundError(
            f"Customer file was not found:\n{CUSTOMER_FILE}\n\n"
            "Run the customer-generation cell first."
        )

    customers_by_cif = {}
    customer_rows = []

    with open(
        CUSTOMER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_NO",
            "CUSTOMER_TYPE",
            "PIN",
            "CREATION_DT"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "Missing customer columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_no = (
                row.get("CUSTOMER_NO") or ""
            ).strip()

            if not customer_no:
                continue

            customers_by_cif[customer_no] = row
            customer_rows.append(row)

    return customers_by_cif, customer_rows


def create_inquiry_customer_list(customer_rows):
    """
    Creates exactly 40,000 inquiry records.

    Every one of the 15,000 customers appears at least once.
    The remaining 25,000 inquiries are distributed across customers,
    so some customers have multiple credit-history inquiries.
    """

    if len(customer_rows) > NUMBER_OF_MKR_RECORDS:
        raise ValueError(
            "There are more customers than the requested "
            "number of MKR records."
        )

    inquiry_customers = customer_rows.copy()

    additional_inquiry_count = (
        NUMBER_OF_MKR_RECORDS - len(customer_rows)
    )

    weights = []

    for customer_row in customer_rows:
        customer_type = get_customer_type(customer_row)

        # Individuals are more likely to make repeated applications.
        if customer_type == "I":
            weights.append(1.8)

        elif customer_type == "C":
            weights.append(1.3)

        else:
            weights.append(0.7)

    additional_customers = random.choices(
        population=customer_rows,
        weights=weights,
        k=additional_inquiry_count
    )

    inquiry_customers.extend(additional_customers)

    # Mix the records so repeated inquiries are not next to each other.
    random.shuffle(inquiry_customers)

    return inquiry_customers


def create_mkr_row(customer_row, sequence_number):
    """
    Creates one MKR inquiry row.

    Individual:
        PIN_CODE is filled
        DOCUMENT_NO is blank

    Company / Bank:
        PIN_CODE is blank
        DOCUMENT_NO contains CUSTOMER_NO / CIF
    """

    customer_type = get_customer_type(customer_row)

    customer_creation_date = parse_date(
        customer_row.get("CREATION_DT") or ""
    )

    reporting_start_date = max(
        customer_creation_date or date(2018, 1, 1),
        date(2018, 1, 1)
    )

    reporting_end_date = date(2026, 6, 20)

    reporting_date = random_date(
        start_date=reporting_start_date,
        end_date=reporting_end_date
    )

    insert_date = reporting_date + timedelta(
        days=random.randint(0, 5)
    )

    search_id = (
        f"MKRM{reporting_date:%y}{sequence_number:010d}"
    )

    report_id = (
        f"RPT{reporting_date:%y%m}{sequence_number:09d}"
    )

    if customer_type == "I":
        pin_code = (customer_row.get("PIN") or "").strip()
        document_no = ""

        if not pin_code:
            raise ValueError(
                "An individual customer has no PIN. "
                f"CIF: {customer_row.get('CUSTOMER_NO')}"
            )

    else:
        pin_code = ""
        document_no = (
            customer_row.get("CUSTOMER_NO") or ""
        ).strip()

        if not document_no:
            raise ValueError(
                "A company/bank has no CUSTOMER_NO / CIF."
            )

    balance = generate_balance(customer_type)

    return {
        "SEARCH_ID": search_id,
        "PIN_CODE": pin_code,
        "BALANCE": f"{balance:.2f}",
        "REPORTID": report_id,
        "REPORTINGDATE": reporting_date.isoformat(),
        "INSERT_DATE": insert_date.isoformat(),
        "DOCUMENT_NO": document_no
    }


def validate_mkr_rows(mkr_rows, customers_by_cif):
    """Checks the row count, keys, and customer-table join logic."""

    search_ids = [
        row["SEARCH_ID"]
        for row in mkr_rows
    ]

    report_ids = [
        row["REPORTID"]
        for row in mkr_rows
    ]

    customer_pins = {
        (customer.get("PIN") or "").strip()
        for customer in customers_by_cif.values()
        if get_customer_type(customer) == "I"
    }

    customer_cifs_for_companies = {
        (customer.get("CUSTOMER_NO") or "").strip()
        for customer in customers_by_cif.values()
        if get_customer_type(customer) in {"C", "B"}
    }

    if len(mkr_rows) != NUMBER_OF_MKR_RECORDS:
        raise ValueError(
            f"Expected {NUMBER_OF_MKR_RECORDS:,} records, "
            f"but generated {len(mkr_rows):,}."
        )

    if len(search_ids) != len(set(search_ids)):
        raise ValueError("Duplicate SEARCH_ID values were generated.")

    if len(report_ids) != len(set(report_ids)):
        raise ValueError("Duplicate REPORTID values were generated.")

    for row in mkr_rows:
        pin_code = row["PIN_CODE"]
        document_no = row["DOCUMENT_NO"]

        if pin_code and document_no:
            raise ValueError(
                "One MKR record incorrectly contains both "
                "PIN_CODE and DOCUMENT_NO."
            )

        if not pin_code and not document_no:
            raise ValueError(
                "One MKR record contains neither PIN_CODE "
                "nor DOCUMENT_NO."
            )

        if pin_code and pin_code not in customer_pins:
            raise ValueError(
                f"PIN_CODE {pin_code} does not match a customer."
            )

        if document_no and document_no not in customer_cifs_for_companies:
            raise ValueError(
                f"DOCUMENT_NO {document_no} does not match "
                "a company/bank CIF."
            )


def write_csv(file_path, rows):
    """Writes data to a CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_MASTER_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_master():
    customers_by_cif, customer_rows = load_customers()

    inquiry_customers = create_inquiry_customer_list(
        customer_rows
    )

    mkr_rows = []

    for sequence_number, customer_row in enumerate(
        inquiry_customers,
        start=1
    ):
        mkr_rows.append(
            create_mkr_row(
                customer_row=customer_row,
                sequence_number=sequence_number
            )
        )

    validate_mkr_rows(
        mkr_rows=mkr_rows,
        customers_by_cif=customers_by_cif
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_master_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER / "rs_mkr_master_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_master_generation_summary.json"
    )

    write_csv(full_output_file, mkr_rows)
    write_csv(sample_output_file, mkr_rows[:100])

    individual_inquiries = sum(
        1 for row in mkr_rows
        if row["PIN_CODE"]
    )

    corporate_or_bank_inquiries = sum(
        1 for row in mkr_rows
        if row["DOCUMENT_NO"]
    )

    summary = {
        "table_name": "RS_MKR_MASTER",
        "generated_rows": len(mkr_rows),
        "unique_search_ids": len(
            {row["SEARCH_ID"] for row in mkr_rows}
        ),
        "unique_report_ids": len(
            {row["REPORTID"] for row in mkr_rows}
        ),
        "individual_inquiries_joined_by_pin": individual_inquiries,
        "company_bank_inquiries_joined_by_cif": (
            corporate_or_bank_inquiries
        ),
        "join_rules": {
            "individual": (
                "RS_MKR_MASTER.PIN_CODE = RS_CUSTOMERS.PIN"
            ),
            "company_or_bank": (
                "RS_MKR_MASTER.DOCUMENT_NO "
                "= RS_CUSTOMERS.CUSTOMER_NO"
            )
        },
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR master generation completed successfully.")
    print(f"Generated inquiry rows: {len(mkr_rows):,}")
    print(f"Individual inquiries by PIN: {individual_inquiries:,}")
    print(
        "Company/bank inquiries by CIF: "
        f"{corporate_or_bank_inquiries:,}"
    )
    print("Unmatched customer keys: 0")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_master()

MKR master generation completed successfully.
Generated inquiry rows: 40,000
Individual inquiries by PIN: 31,616
Company/bank inquiries by CIF: 8,384
Unmatched customer keys: 0
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_master_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_master_base_sample.csv


In [2]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260626
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# LIABILITIES TABLE COLUMNS
# ============================================================

MKR_LIABILITY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "ACCOUNTNO",
    "BANKID",
    "BANKNAME",
    "COLLATERALANYINFO",
    "COLLATERALCODE",
    "COLLATERALMARKETVALUE",
    "COLLATERALREGISTRYAGENCY",
    "COLLATERALREGISTRYDATE",
    "COLLATERALREGISTRYNO",
    "COLLATERALTYPENAME",
    "CREDITPURPOSE",
    "CREDITPURPOSENAME",
    "CREDITSTATUS",
    "CREDITSTATUSCLOSEDATE",
    "CREDITSTATUSNAME",
    "CREDITTYPE",
    "CREDITTYPENAME",
    "CURRENCY",
    "CURRENCYNAME",
    "DAYSINTERESTOVERDUE",
    "DAYSMAINSUMOVERDUE",
    "GRANTEDON",
    "ID",
    "INITIALAMOUNT",
    "INTERESTRATE",
    "LASTPAYMENTDATE",
    "LASTUPDATEDDATE",
    "LINEAMOUNT",
    "MKRID",
    "MONTHLYPAYMENTAMOUNT",
    "ORGIDTYPE",
    "OUTSTANDINGDEBTINTEREST",
    "OUTSTANDINGDEBTMAIN",
    "PROLONGATIONS",
    "CONTRACTDUEON",
    "INS_DTTM"
]


# ============================================================
# REFERENCE DATA
# ============================================================

BANKS = [
    ("001", "Azer Finance Bank"),
    ("002", "Caspian Commercial Bank"),
    ("003", "Baku Credit Bank"),
    ("004", "Shirvan Bank"),
    ("005", "Khazar Finance"),
    ("006", "Atlas Bank"),
    ("007", "Regional Development Bank")
]

INDIVIDUAL_CREDIT_TYPES = [
    {
        "code": "001",
        "name": "Consumer Loan",
        "purpose_code": "001",
        "purpose_name": "Consumer Expenses",
        "term_min": 12,
        "term_max": 60,
        "secured": False,
        "line": False
    },
    {
        "code": "002",
        "name": "Mortgage Loan",
        "purpose_code": "002",
        "purpose_name": "Property Purchase",
        "term_min": 60,
        "term_max": 240,
        "secured": True,
        "collateral_type": "Residential Property",
        "collateral_code": "001",
        "line": False
    },
    {
        "code": "003",
        "name": "Auto Loan",
        "purpose_code": "003",
        "purpose_name": "Vehicle Purchase",
        "term_min": 24,
        "term_max": 84,
        "secured": True,
        "collateral_type": "Vehicle",
        "collateral_code": "002",
        "line": False
    },
    {
        "code": "004",
        "name": "Credit Card",
        "purpose_code": "004",
        "purpose_name": "Card Spending",
        "term_min": 12,
        "term_max": 48,
        "secured": False,
        "line": True
    }
]

CORPORATE_CREDIT_TYPES = [
    {
        "code": "005",
        "name": "SME Business Loan",
        "purpose_code": "005",
        "purpose_name": "Business Development",
        "term_min": 12,
        "term_max": 84,
        "secured": True,
        "collateral_type": "Commercial Property",
        "collateral_code": "003",
        "line": False
    },
    {
        "code": "006",
        "name": "Business Credit Line",
        "purpose_code": "006",
        "purpose_name": "Working Capital",
        "term_min": 12,
        "term_max": 60,
        "secured": False,
        "line": True
    },
    {
        "code": "007",
        "name": "Equipment Financing",
        "purpose_code": "007",
        "purpose_name": "Equipment Purchase",
        "term_min": 24,
        "term_max": 72,
        "secured": True,
        "collateral_type": "Equipment",
        "collateral_code": "007",
        "line": False
    },
    {
        "code": "008",
        "name": "Commercial Mortgage",
        "purpose_code": "008",
        "purpose_name": "Commercial Property Purchase",
        "term_min": 60,
        "term_max": 180,
        "secured": True,
        "collateral_type": "Commercial Property",
        "collateral_code": "003",
        "line": False
    }
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between two dates."""

    if start_date >= end_date:
        return start_date

    number_of_days = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, number_of_days)
    )


def to_decimal(value):
    """Converts a numeric text value into Decimal safely."""

    try:
        return Decimal(str(value)).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

    except Exception:
        return Decimal("0.00")


def format_decimal(value):
    """Formats Decimal as text with exactly two decimals."""

    return format(
        value.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        ),
        "f"
    )


def choose_liability_count(balance, is_individual):
    """
    Determines how many child liabilities one inquiry has.

    Small balances usually have one liability.
    Larger balances can have several liabilities.
    """

    if balance < Decimal("1000"):
        return 1

    if is_individual:
        if balance < Decimal("5000"):
            return random.choices(
                [1, 2],
                weights=[75, 25],
                k=1
            )[0]

        return random.choices(
            [1, 2, 3, 4],
            weights=[30, 40, 22, 8],
            k=1
        )[0]

    return random.choices(
        [1, 2, 3, 4, 5],
        weights=[15, 30, 30, 18, 7],
        k=1
    )[0]


def split_balance(total_balance, number_of_liabilities):
    """
    Splits one MKR master BALANCE into child liability balances.

    The final child balances always sum exactly to the parent BALANCE.
    """

    total_cents = int(total_balance * 100)

    if number_of_liabilities == 1:
        return [total_cents]

    minimum_cents_per_liability = 100
    minimum_required = (
        number_of_liabilities
        * minimum_cents_per_liability
    )

    if total_cents <= minimum_required:
        return [total_cents]

    remaining_cents = total_cents - minimum_required

    weights = [
        random.uniform(0.4, 1.6)
        for _ in range(number_of_liabilities)
    ]

    total_weight = sum(weights)

    allocations = [
        minimum_cents_per_liability
        + int(
            remaining_cents
            * weight
            / total_weight
        )
        for weight in weights
    ]

    difference = total_cents - sum(allocations)

    allocations[-1] += difference

    return allocations


def choose_credit_status():
    """
    Generates a realistic active-liability status.

    All generated child liabilities have an outstanding balance,
    therefore closed status is not used here.
    """

    return random.choices(
        [
            ("001", "Active", 0),
            ("005", "Past Due", random.randint(1, 180)),
            ("007", "Restructured", random.randint(0, 60))
        ],
        weights=[78, 16, 6],
        k=1
    )[0]


def choose_credit_type(is_individual):
    """Chooses an appropriate credit type."""

    if is_individual:
        return random.choices(
            INDIVIDUAL_CREDIT_TYPES,
            weights=[44, 12, 17, 27],
            k=1
        )[0]

    return random.choices(
        CORPORATE_CREDIT_TYPES,
        weights=[38, 28, 17, 17],
        k=1
    )[0]


def generate_credit_dates(
    reporting_date,
    status_code,
    credit_type
):
    """
    Creates dates that make business sense:
    - active credit normally ends after reporting date
    - past-due credit may be near/after maturity
    """

    term_months = random.randint(
        credit_type["term_min"],
        credit_type["term_max"]
    )

    term_days = term_months * 30

    if status_code == "001":
        maximum_age = max(30, term_days - 30)

        granted_on = reporting_date - timedelta(
            days=random.randint(30, maximum_age)
        )

    else:
        granted_on = reporting_date - timedelta(
            days=random.randint(
                term_days - 60,
                term_days + 730
            )
        )

    contract_due_on = granted_on + timedelta(
        days=term_days
    )

    last_payment_start = max(
        granted_on,
        reporting_date - timedelta(days=180)
    )

    last_payment_date = random_date(
        last_payment_start,
        reporting_date
    )

    return (
        granted_on,
        contract_due_on,
        last_payment_date
    )


def generate_account_number(owner_key, liability_slot):
    """
    Creates a repeated external-account identifier for a customer.

    The same customer can see the same liability account
    in multiple later MKR inquiries.
    """

    safe_owner_key = (
        owner_key.replace(" ", "")
        .replace("-", "")
        .upper()
    )

    return f"EXT{safe_owner_key[-12:]}{liability_slot:02d}"


# ============================================================
# INPUT DATA
# ============================================================

def load_mkr_master():
    """Loads the parent MKR master file."""

    if not MKR_MASTER_FILE.exists():
        raise FileNotFoundError(
            f"MKR master file was not found:\n{MKR_MASTER_FILE}\n\n"
            "Run the MKR master generation cell first."
        )

    master_rows = []

    with open(
        MKR_MASTER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "SEARCH_ID",
            "PIN_CODE",
            "BALANCE",
            "REPORTINGDATE",
            "INSERT_DATE",
            "DOCUMENT_NO"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "MKR master is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            master_rows.append(row)

    return master_rows


# ============================================================
# ROW GENERATION
# ============================================================

def create_liability_rows_for_master(
    master_row,
    starting_id
):
    """
    Creates zero or more liabilities for one MKR inquiry.

    Zero BALANCE means the customer had no reported active liabilities,
    so no child liability rows are created.
    """

    search_id = (
        master_row.get("SEARCH_ID") or ""
    ).strip()

    pin_code = (
        master_row.get("PIN_CODE") or ""
    ).strip()

    document_no = (
        master_row.get("DOCUMENT_NO") or ""
    ).strip()

    reporting_date = parse_date(
        master_row.get("REPORTINGDATE") or ""
    )

    insert_date = parse_date(
        master_row.get("INSERT_DATE") or ""
    )

    parent_balance = to_decimal(
        master_row.get("BALANCE") or "0"
    )

    if not search_id:
        raise ValueError("MKR master contains blank SEARCH_ID.")

    if not reporting_date:
        raise ValueError(
            f"Invalid REPORTINGDATE for SEARCH_ID: {search_id}"
        )

    if parent_balance <= Decimal("0.00"):
        return []

    is_individual = bool(pin_code)

    owner_key = pin_code if is_individual else document_no

    if not owner_key:
        raise ValueError(
            f"SEARCH_ID {search_id} has no PIN_CODE "
            "and no DOCUMENT_NO."
        )

    liability_count = choose_liability_count(
        balance=parent_balance,
        is_individual=is_individual
    )

    balance_allocations = split_balance(
        total_balance=parent_balance,
        number_of_liabilities=liability_count
    )

    liability_rows = []

    for liability_slot, total_debt_cents in enumerate(
        balance_allocations,
        start=1
    ):
        total_debt = (
            Decimal(total_debt_cents)
            / Decimal("100")
        )

        (
            status_code,
            status_name,
            overdue_days
        ) = choose_credit_status()

        credit_type = choose_credit_type(
            is_individual=is_individual
        )

        bank_id, bank_name = random.choice(BANKS)

        (
            granted_on,
            contract_due_on,
            last_payment_date
        ) = generate_credit_dates(
            reporting_date=reporting_date,
            status_code=status_code,
            credit_type=credit_type
        )

        if overdue_days == 0:
            interest_ratio = Decimal(
                str(random.uniform(0.01, 0.08))
            )
        else:
            interest_ratio = Decimal(
                str(random.uniform(0.05, 0.20))
            )

        interest_debt = (
            total_debt * interest_ratio
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        main_debt = total_debt - interest_debt

        initial_multiplier = Decimal(
            str(random.uniform(1.10, 3.20))
        )

        initial_amount = (
            total_debt * initial_multiplier
        ).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

        interest_rate = round(
            random.uniform(9.0, 28.0),
            2
        )

        if credit_type["line"]:
            line_amount = (
                initial_amount * Decimal(
                    str(random.uniform(1.05, 1.40))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            monthly_payment = (
                total_debt * Decimal(
                    str(random.uniform(0.05, 0.15))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

        else:
            line_amount = Decimal("0.00")

            monthly_payment = max(
                Decimal("20.00"),
                (
                    total_debt * Decimal(
                        str(random.uniform(0.03, 0.12))
                    )
                ).quantize(
                    Decimal("0.01"),
                    rounding=ROUND_HALF_UP
                )
            )

        is_secured = credit_type.get(
            "secured",
            False
        )

        if is_secured:
            collateral_value = (
                initial_amount * Decimal(
                    str(random.uniform(1.05, 1.50))
                )
            ).quantize(
                Decimal("0.01"),
                rounding=ROUND_HALF_UP
            )

            collateral_code = credit_type.get(
                "collateral_code",
                "007"
            )

            collateral_type = credit_type.get(
                "collateral_type",
                "Other Collateral"
            )

            collateral_registry_date = (
                granted_on
                + timedelta(days=random.randint(0, 30))
            )

            collateral_registry_no = (
                f"COL{granted_on:%y}{starting_id + liability_slot:09d}"
            )

            collateral_registry_agency = (
                "STATE COLLATERAL REGISTRY"
            )

            collateral_any_info = (
                f"{collateral_type.upper()} SECURES CREDIT"
            )

        else:
            collateral_value = Decimal("0.00")
            collateral_code = "000"
            collateral_type = ""
            collateral_registry_date = ""
            collateral_registry_no = ""
            collateral_registry_agency = ""
            collateral_any_info = ""

        record_id = str(
            starting_id + liability_slot
        )

        updated_date = reporting_date

        created_timestamp_date = (
            insert_date
            if insert_date
            else reporting_date
        )

        created_timestamp = datetime.combine(
            created_timestamp_date,
            datetime.min.time()
        ) + timedelta(
            hours=random.randint(8, 19),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        liability_rows.append({
            "SEARCH_ID": search_id,
            "PIN_CODE": pin_code,
            "ACCOUNTNO": generate_account_number(
                owner_key=owner_key,
                liability_slot=liability_slot
            ),
            "BANKID": bank_id,
            "BANKNAME": bank_name,
            "COLLATERALANYINFO": collateral_any_info,
            "COLLATERALCODE": collateral_code,
            "COLLATERALMARKETVALUE": format_decimal(
                collateral_value
            ),
            "COLLATERALREGISTRYAGENCY": (
                collateral_registry_agency
            ),
            "COLLATERALREGISTRYDATE": (
                collateral_registry_date.isoformat()
                if collateral_registry_date
                else ""
            ),
            "COLLATERALREGISTRYNO": collateral_registry_no,
            "COLLATERALTYPENAME": collateral_type,
            "CREDITPURPOSE": credit_type["purpose_code"],
            "CREDITPURPOSENAME": credit_type["purpose_name"],
            "CREDITSTATUS": status_code,
            "CREDITSTATUSCLOSEDATE": "",
            "CREDITSTATUSNAME": status_name,
            "CREDITTYPE": credit_type["code"],
            "CREDITTYPENAME": credit_type["name"],
            "CURRENCY": "AZN",
            "CURRENCYNAME": "Azerbaijani Manat",
            "DAYSINTERESTOVERDUE": str(overdue_days),
            "DAYSMAINSUMOVERDUE": str(overdue_days),
            "GRANTEDON": granted_on.isoformat(),
            "ID": record_id,
            "INITIALAMOUNT": format_decimal(initial_amount),
            "INTERESTRATE": str(interest_rate),
            "LASTPAYMENTDATE": last_payment_date.isoformat(),
            "LASTUPDATEDDATE": updated_date.isoformat(),
            "LINEAMOUNT": format_decimal(line_amount),
            "MKRID": f"MKR-{record_id}",
            "MONTHLYPAYMENTAMOUNT": format_decimal(
                monthly_payment
            ),
            "ORGIDTYPE": "001",
            "OUTSTANDINGDEBTINTEREST": format_decimal(
                interest_debt
            ),
            "OUTSTANDINGDEBTMAIN": format_decimal(
                main_debt
            ),
            "PROLONGATIONS": str(
                random.randint(1, 3)
                if status_code == "007"
                else random.choices(
                    [0, 1],
                    weights=[92, 8],
                    k=1
                )[0]
            ),
            "CONTRACTDUEON": contract_due_on.isoformat(),
            "INS_DTTM": created_timestamp.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        })

    return liability_rows


# ============================================================
# VALIDATION
# ============================================================

def validate_liabilities(
    master_rows,
    liability_rows
):
    """
    Validates:
    1. Every child SEARCH_ID exists in MKR master.
    2. Liability PIN matches the parent inquiry PIN.
    3. Child principal + interest sums to parent BALANCE.
    4. Every positive-balance parent has at least one liability.
    """

    master_by_search_id = {
        row["SEARCH_ID"].strip(): row
        for row in master_rows
    }

    balances_by_search_id = defaultdict(
        lambda: Decimal("0.00")
    )

    liability_search_ids = set()
    liability_ids = set()

    for row in liability_rows:
        search_id = row["SEARCH_ID"].strip()

        if search_id not in master_by_search_id:
            raise ValueError(
                f"Liability SEARCH_ID does not exist in master: "
                f"{search_id}"
            )

        parent_pin = (
            master_by_search_id[search_id]
            .get("PIN_CODE", "")
            .strip()
        )

        if row["PIN_CODE"].strip() != parent_pin:
            raise ValueError(
                f"PIN_CODE mismatch for SEARCH_ID: {search_id}"
            )

        if row["ID"] in liability_ids:
            raise ValueError(
                f"Duplicate liability ID found: {row['ID']}"
            )

        liability_ids.add(row["ID"])
        liability_search_ids.add(search_id)

        row_total = (
            to_decimal(row["OUTSTANDINGDEBTMAIN"])
            + to_decimal(row["OUTSTANDINGDEBTINTEREST"])
        )

        balances_by_search_id[search_id] += row_total

    for search_id, master_row in master_by_search_id.items():
        parent_balance = to_decimal(
            master_row.get("BALANCE") or "0"
        )

        child_balance = balances_by_search_id[search_id]

        if parent_balance > Decimal("0.00"):
            if search_id not in liability_search_ids:
                raise ValueError(
                    "Positive-balance MKR master has no child "
                    f"liability: {search_id}"
                )

            if child_balance != parent_balance:
                raise ValueError(
                    f"Balance mismatch for SEARCH_ID {search_id}. "
                    f"Master: {parent_balance}; "
                    f"Liabilities: {child_balance}"
                )

        else:
            if child_balance != Decimal("0.00"):
                raise ValueError(
                    f"Zero-balance MKR master has liabilities: "
                    f"{search_id}"
                )


def write_csv(file_path, rows):
    """Writes liabilities to CSV."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_LIABILITY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_liabilities():
    master_rows = load_mkr_master()

    liability_rows = []
    next_id = 1

    for master_row in master_rows:
        new_rows = create_liability_rows_for_master(
            master_row=master_row,
            starting_id=next_id
        )

        liability_rows.extend(new_rows)
        next_id += len(new_rows)

    validate_liabilities(
        master_rows=master_rows,
        liability_rows=liability_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER / "rs_mkr_liabilities_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_liabilities_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liabilities_generation_summary.json"
    )

    write_csv(full_output_file, liability_rows)
    write_csv(sample_output_file, liability_rows[:100])

    unique_parent_search_ids = len(
        {row["SEARCH_ID"] for row in liability_rows}
    )

    total_master_records = len(master_rows)

    summary = {
        "table_name": "RS_MKR_LIABILITIES",
        "generated_rows": len(liability_rows),
        "unique_liability_ids": len(
            {row["ID"] for row in liability_rows}
        ),
        "master_search_ids": total_master_records,
        "master_search_ids_with_liabilities": (
            unique_parent_search_ids
        ),
        "join_rule": (
            "RS_MKR_LIABILITIES.SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "balance_rule": (
            "For every positive-balance SEARCH_ID, "
            "sum(OUTSTANDINGDEBTMAIN + "
            "OUTSTANDINGDEBTINTEREST) = "
            "RS_MKR_MASTER.BALANCE"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR liabilities generation completed successfully.")
    print(f"Generated liability rows: {len(liability_rows):,}")
    print(
        "MKR master search IDs with liabilities: "
        f"{unique_parent_search_ids:,}"
    )
    print(
        "MKR master search IDs without liabilities "
        "(zero balance): "
        f"{total_master_records - unique_parent_search_ids:,}"
    )
    print("Unmatched SEARCH_ID values: 0")
    print("Parent-to-child balance validation: passed")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_liabilities()

MKR liabilities generation completed successfully.
Generated liability rows: 70,921
MKR master search IDs with liabilities: 33,666
MKR master search IDs without liabilities (zero balance): 6,334
Unmatched SEARCH_ID values: 0
Parent-to-child balance validation: passed
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_liabilities_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_liabilities_base_sample.csv


In [3]:
import csv
import json
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

MKR_LIABILITIES_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_liabilities_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# LIABILITY HISTORY TABLE COLUMNS
# ============================================================

MKR_LIABILITY_HISTORY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "LIABILITY_ID",
    "OVERDUEDAYS",
    "REPORTINGPERIOD",
    "CREDIT_STATUS",
    "INS_DTTM"
]


# ============================================================
# FILE READING
# ============================================================

def load_csv_rows(file_path, required_columns, table_name):
    """Reads a CSV file and checks required columns."""

    if not file_path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{file_path}"
        )

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        available_columns = set(reader.fieldnames or [])
        missing_columns = required_columns - available_columns

        if missing_columns:
            raise ValueError(
                f"{table_name} is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        return list(reader)


# ============================================================
# HISTORY ROW CREATION
# ============================================================

def create_history_row(master_row, liability_row):
    """
    Creates one liability-history snapshot.

    Values are copied from the matching master and liability records
    so all reporting dates, overdue days, and statuses stay consistent.
    """

    return {
        "SEARCH_ID": liability_row["SEARCH_ID"].strip(),
        "PIN_CODE": liability_row.get("PIN_CODE", "").strip(),
        "LIABILITY_ID": liability_row["ID"].strip(),
        "OVERDUEDAYS": liability_row[
            "DAYSMAINSUMOVERDUE"
        ].strip(),
        "REPORTINGPERIOD": master_row[
            "REPORTINGDATE"
        ].strip(),
        "CREDIT_STATUS": liability_row[
            "CREDITSTATUS"
        ].strip(),
        "INS_DTTM": liability_row["INS_DTTM"].strip()
    }


# ============================================================
# VALIDATION
# ============================================================

def validate_and_create_history(master_rows, liability_rows):
    """
    Validates parent-child relationships, then creates history rows.

    Rules:
    1. Every liability SEARCH_ID exists in MKR master.
    2. Liability PIN_CODE equals the parent master PIN_CODE.
    3. SEARCH_ID + LIABILITY_ID is unique in history.
    4. Every liability receives exactly one history row.
    """

    master_by_search_id = {}

    for master_row in master_rows:
        search_id = master_row["SEARCH_ID"].strip()

        if not search_id:
            raise ValueError(
                "MKR master contains a blank SEARCH_ID."
            )

        if search_id in master_by_search_id:
            raise ValueError(
                f"Duplicate SEARCH_ID in MKR master: {search_id}"
            )

        master_by_search_id[search_id] = master_row

    history_rows = []
    history_keys = set()

    for liability_row in liability_rows:
        search_id = liability_row["SEARCH_ID"].strip()
        liability_id = liability_row["ID"].strip()

        if search_id not in master_by_search_id:
            raise ValueError(
                "Liability has no matching parent MKR master record. "
                f"SEARCH_ID: {search_id}"
            )

        if not liability_id:
            raise ValueError(
                f"Blank liability ID for SEARCH_ID: {search_id}"
            )

        master_row = master_by_search_id[search_id]

        master_pin = master_row.get(
            "PIN_CODE",
            ""
        ).strip()

        liability_pin = liability_row.get(
            "PIN_CODE",
            ""
        ).strip()

        if master_pin != liability_pin:
            raise ValueError(
                "PIN mismatch between MKR master and liability. "
                f"SEARCH_ID: {search_id}"
            )

        history_key = (search_id, liability_id)

        if history_key in history_keys:
            raise ValueError(
                "Duplicate SEARCH_ID + LIABILITY_ID combination: "
                f"{search_id} / {liability_id}"
            )

        history_keys.add(history_key)

        history_rows.append(
            create_history_row(
                master_row=master_row,
                liability_row=liability_row
            )
        )

    if len(history_rows) != len(liability_rows):
        raise ValueError(
            "History row count does not match liability row count."
        )

    return history_rows


# ============================================================
# FILE WRITING
# ============================================================

def write_csv(file_path, rows):
    """Writes rows into a CSV file."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_LIABILITY_HISTORY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_liability_history():
    master_rows = load_csv_rows(
        file_path=MKR_MASTER_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "REPORTINGDATE"
        },
        table_name="MKR master"
    )

    liability_rows = load_csv_rows(
        file_path=MKR_LIABILITIES_FILE,
        required_columns={
            "SEARCH_ID",
            "PIN_CODE",
            "ID",
            "DAYSMAINSUMOVERDUE",
            "CREDITSTATUS",
            "INS_DTTM"
        },
        table_name="MKR liabilities"
    )

    history_rows = validate_and_create_history(
        master_rows=master_rows,
        liability_rows=liability_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liability_history_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_liability_history_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_liability_history_generation_summary.json"
    )

    write_csv(full_output_file, history_rows)
    write_csv(sample_output_file, history_rows[:100])

    summary = {
        "table_name": "RS_MKR_LIABILITY_HISTORY",
        "generated_rows": len(history_rows),
        "mkr_liability_rows": len(liability_rows),
        "row_count_matches_liabilities": (
            len(history_rows) == len(liability_rows)
        ),
        "join_rule": (
            "RS_MKR_LIABILITY_HISTORY.SEARCH_ID "
            "= RS_MKR_LIABILITIES.SEARCH_ID "
            "AND RS_MKR_LIABILITY_HISTORY.LIABILITY_ID "
            "= RS_MKR_LIABILITIES.ID"
        ),
        "reporting_period_rule": (
            "REPORTINGPERIOD = RS_MKR_MASTER.REPORTINGDATE"
        ),
        "overdue_days_rule": (
            "OVERDUEDAYS = RS_MKR_LIABILITIES.DAYSMAINSUMOVERDUE"
        ),
        "credit_status_rule": (
            "CREDIT_STATUS = RS_MKR_LIABILITIES.CREDITSTATUS"
        )
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR liability history generation completed successfully.")
    print(f"Generated history rows: {len(history_rows):,}")
    print(
        "Liability-to-history row count match: passed"
    )
    print("Unmatched SEARCH_ID values: 0")
    print(
        "SEARCH_ID + LIABILITY_ID uniqueness validation: passed"
    )
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_liability_history()

MKR liability history generation completed successfully.
Generated history rows: 70,921
Liability-to-history row count match: passed
Unmatched SEARCH_ID values: 0
SEARCH_ID + LIABILITY_ID uniqueness validation: passed
Full file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\raw\rs_mkr_liability_history_base.csv
Sample file: c:\Users\ASUS\Desktop\project\risk_scoring_project\data\samples\rs_mkr_liability_history_base_sample.csv


In [ ]:
import csv
import json
import random
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 20260627
LOOKBACK_DAYS = 730

random.seed(RANDOM_SEED)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

CUSTOMER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_customers_base.csv"
)

MKR_MASTER_FILE = (
    PROJECT_ROOT / "data" / "raw" / "rs_mkr_master_base.csv"
)

RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
SAMPLE_DATA_FOLDER = PROJECT_ROOT / "data" / "samples"

RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# INQUIRY HISTORY TABLE COLUMNS
# ============================================================

MKR_INQUIRY_HISTORY_COLUMNS = [
    "SEARCH_ID",
    "PIN_CODE",
    "INQBANKID",
    "INQBANKNAME",
    "INQDATE",
    "INQORGIDTYPE",
    "INQPURPOSEID",
    "INQPURPOSENAME",
    "INS_DTTM",
    "DOCUMENTNO",
    "REPORTINGDATE"
]


# ============================================================
# REFERENCE DATA
# ============================================================

INQUIRY_BANKS = [
    ("481", "Azərbaycan Beynəlxalq Bankı ASC", "001"),
    ("001", "XXXX", "001"),
    ("002", "XXXX", "001"),
    ("003", "XXXX", "001"),
    ("004", "XXXX", "001"),
    ("005", "", "002"),
    ("006", "", "002"),
    ("007", "", "003")
]

INQUIRY_PURPOSES_INDIVIDUAL = [
    ("001", "Consumer Loan"),
    ("002", "Mortgage Loan"),
    ("003", "Auto Loan"),
    ("004", "Credit Card"),
    ("006", "Credit Line"),
    ("009", "Other")
]

INQUIRY_PURPOSES_CORPORATE = [
    ("005", "Business Loan"),
    ("006", "Working Capital Credit Line"),
    ("007", "Leasing"),
    ("008", "Guarantee"),
    ("009", "Other")
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_date(value):
    """Converts YYYY-MM-DD text into a Python date."""

    if not value:
        return None

    try:
        return datetime.strptime(
            value.strip(),
            "%Y-%m-%d"
        ).date()

    except ValueError:
        return None


def random_date(start_date, end_date):
    """Returns one random date between two dates."""

    if start_date >= end_date:
        return start_date

    days_between = (end_date - start_date).days

    return start_date + timedelta(
        days=random.randint(0, days_between)
    )


def get_owner_key(pin_code, document_no):
    """
    Creates one consistent customer key.

    Individual:
        P:<PIN>

    Company / bank:
        D:<CIF>
    """

    pin_code = (pin_code or "").strip()
    document_no = (document_no or "").strip()

    if pin_code:
        return f"P:{pin_code}"

    if document_no:
        return f"D:{document_no}"

    raise ValueError(
        "Neither PIN_CODE nor DOCUMENT_NO is available."
    )


def choose_inquiry_event(is_individual, inquiry_date, event_number):
    """
    Creates one actual historical inquiry event.

    This event can appear in the history of multiple later
    MKR master searches for the same customer.
    """

    bank_id, bank_name, org_type = random.choice(
        INQUIRY_BANKS
    )

    if is_individual:
        purpose_id, purpose_name = random.choices(
            INQUIRY_PURPOSES_INDIVIDUAL,
            weights=[36, 10, 14, 22, 12, 6],
            k=1
        )[0]

    else:
        purpose_id, purpose_name = random.choices(
            INQUIRY_PURPOSES_CORPORATE,
            weights=[42, 28, 12, 8, 10],
            k=1
        )[0]

    return {
        "event_id": f"INQ_EVENT_{event_number:09d}",
        "inquiry_date": inquiry_date,
        "bank_id": bank_id,
        "bank_name": bank_name,
        "org_type": org_type,
        "purpose_id": purpose_id,
        "purpose_name": purpose_name
    }


# ============================================================
# INPUT DATA
# ============================================================

def load_customers():
    """
    Reads customers and creates a dictionary:
    owner key -> customer creation date.
    """

    if not CUSTOMER_FILE.exists():
        raise FileNotFoundError(
            f"Customer file was not found:\n{CUSTOMER_FILE}"
        )

    customer_creation_dates = {}

    with open(
        CUSTOMER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "CUSTOMER_NO",
            "PIN",
            "CUSTOMER_TYPE",
            "CREATION_DT"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "Customer table is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            customer_type = (
                row.get("CUSTOMER_TYPE") or ""
            ).strip().upper()

            pin_code = (row.get("PIN") or "").strip()
            customer_no = (
                row.get("CUSTOMER_NO") or ""
            ).strip()

            if customer_type == "I":
                owner_key = get_owner_key(
                    pin_code=pin_code,
                    document_no=""
                )
            else:
                owner_key = get_owner_key(
                    pin_code="",
                    document_no=customer_no
                )

            creation_date = parse_date(
                row.get("CREATION_DT") or ""
            )

            customer_creation_dates[owner_key] = (
                creation_date or date(2010, 1, 1)
            )

    return customer_creation_dates


def load_mkr_master():
    """
    Reads MKR master records.

    SEARCH_ID is unique here.
    It will repeat in the inquiry-history child table.
    """

    if not MKR_MASTER_FILE.exists():
        raise FileNotFoundError(
            f"MKR master file was not found:\n{MKR_MASTER_FILE}"
        )

    master_rows = []

    with open(
        MKR_MASTER_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        reader = csv.DictReader(file)

        required_columns = {
            "SEARCH_ID",
            "PIN_CODE",
            "DOCUMENT_NO",
            "REPORTINGDATE",
            "INSERT_DATE"
        }

        missing_columns = required_columns - set(
            reader.fieldnames or []
        )

        if missing_columns:
            raise ValueError(
                "MKR master is missing columns: "
                + ", ".join(sorted(missing_columns))
            )

        seen_search_ids = set()

        for row in reader:
            search_id = (
                row.get("SEARCH_ID") or ""
            ).strip()

            if not search_id:
                raise ValueError(
                    "MKR master contains a blank SEARCH_ID."
                )

            if search_id in seen_search_ids:
                raise ValueError(
                    f"Duplicate SEARCH_ID in MKR master: {search_id}"
                )

            seen_search_ids.add(search_id)

            pin_code = (row.get("PIN_CODE") or "").strip()
            document_no = (
                row.get("DOCUMENT_NO") or ""
            ).strip()

            owner_key = get_owner_key(
                pin_code=pin_code,
                document_no=document_no
            )

            reporting_date = parse_date(
                row.get("REPORTINGDATE") or ""
            )

            insert_date = parse_date(
                row.get("INSERT_DATE") or ""
            )

            if not reporting_date:
                raise ValueError(
                    f"Invalid REPORTINGDATE for {search_id}"
                )

            master_rows.append({
                "SEARCH_ID": search_id,
                "PIN_CODE": pin_code,
                "DOCUMENT_NO": document_no,
                "OWNER_KEY": owner_key,
                "REPORTINGDATE": reporting_date,
                "INSERT_DATE": insert_date or reporting_date
            })

    return master_rows


# ============================================================
# EVENT TIMELINE CREATION
# ============================================================

def build_customer_inquiry_events(
    master_rows,
    customer_creation_dates
):
    """
    Creates one actual inquiry-event timeline per customer.

    The timeline includes:
    1. external historical inquiries
    2. earlier MKR master searches

    A later parent search sees the earlier events from that timeline.
    This prevents random, disconnected inquiry history.
    """

    master_rows_by_owner = defaultdict(list)

    for row in master_rows:
        master_rows_by_owner[row["OWNER_KEY"]].append(row)

    events_by_owner = defaultdict(list)
    event_counter = 1

    for owner_key, owner_master_rows in master_rows_by_owner.items():
        owner_master_rows.sort(
            key=lambda row: (
                row["REPORTINGDATE"],
                row["SEARCH_ID"]
            )
        )

        is_individual = owner_key.startswith("P:")

        first_report_date = owner_master_rows[0][
            "REPORTINGDATE"
        ]

        last_report_date = owner_master_rows[-1][
            "REPORTINGDATE"
        ]

        customer_creation_date = customer_creation_dates.get(
            owner_key,
            date(2010, 1, 1)
        )

        external_history_start = max(
            customer_creation_date,
            first_report_date - timedelta(days=LOOKBACK_DAYS)
        )

        external_history_end = last_report_date - timedelta(days=1)

        # Number of separate external inquiry events for this customer.
        external_event_count = random.choices(
            [0, 1, 2, 3, 4, 5, 6, 7],
            weights=[6, 12, 19, 23, 19, 12, 6, 3],
            k=1
        )[0]

        # Customers with more MKR searches are more likely
        # to have an extensive inquiry history.
        external_event_count += max(
            0,
            len(owner_master_rows) - 2
        )

        external_event_count = min(
            external_event_count,
            10
        )

        # Create external historical inquiry events.
        for _ in range(external_event_count):
            inquiry_date = random_date(
                start_date=external_history_start,
                end_date=external_history_end
            )

            event = choose_inquiry_event(
                is_individual=is_individual,
                inquiry_date=inquiry_date,
                event_number=event_counter
            )

            events_by_owner[owner_key].append(event)
            event_counter += 1

        # Each current master search becomes a possible
        # historical event for future searches by that customer.
        for master_row in owner_master_rows:
            event = choose_inquiry_event(
                is_individual=is_individual,
                inquiry_date=master_row["REPORTINGDATE"],
                event_number=event_counter
            )

            event["source_search_id"] = master_row["SEARCH_ID"]

            events_by_owner[owner_key].append(event)
            event_counter += 1

        events_by_owner[owner_key].sort(
            key=lambda event: (
                event["inquiry_date"],
                event["event_id"]
            )
        )

    return events_by_owner


# ============================================================
# CHILD ROW CREATION
# ============================================================

def create_inquiry_history_rows(
    master_rows,
    events_by_owner
):
    """
    For each current MKR master search:
    - find all past inquiry events
    - keep only the previous 730 days
    - create one child row for each event
    """

    history_rows = []

    for master_row in master_rows:
        search_id = master_row["SEARCH_ID"]
        pin_code = master_row["PIN_CODE"]
        document_no = master_row["DOCUMENT_NO"]
        owner_key = master_row["OWNER_KEY"]
        reporting_date = master_row["REPORTINGDATE"]
        insert_date = master_row["INSERT_DATE"]

        lookback_start = reporting_date - timedelta(
            days=LOOKBACK_DAYS
        )

        eligible_events = [
            event
            for event in events_by_owner[owner_key]
            if (
                lookback_start <= event["inquiry_date"]
                and event["inquiry_date"] < reporting_date
            )
        ]

        for event in eligible_events:
            inserted_at = datetime.combine(
                insert_date,
                datetime.min.time()
            ) + timedelta(
                hours=random.randint(7, 20),
                minutes=random.randint(0, 59),
                seconds=random.randint(0, 59)
            )

            history_rows.append({
                "SEARCH_ID": search_id,
                "PIN_CODE": pin_code,
                "INQBANKID": event["bank_id"],
                "INQBANKNAME": event["bank_name"],
                "INQDATE": event[
                    "inquiry_date"
                ].isoformat(),
                "INQORGIDTYPE": event["org_type"],
                "INQPURPOSEID": event["purpose_id"],
                "INQPURPOSENAME": event[
                    "purpose_name"
                ],
                "INS_DTTM": inserted_at.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
                "DOCUMENTNO": document_no,
                "REPORTINGDATE": reporting_date.isoformat()
            })

    return history_rows


# ============================================================
# VALIDATION
# ============================================================

def validate_inquiry_history(
    master_rows,
    history_rows
):
    """
    Validates:
    1. Every child SEARCH_ID exists in MKR master.
    2. PIN_CODE / DOCUMENTNO match the parent search.
    3. Historical inquiry date is before current report date.
    4. Historical inquiry date is within 730 days.
    """

    master_by_search_id = {
        row["SEARCH_ID"]: row
        for row in master_rows
    }

    rows_per_search = defaultdict(int)

    for history_row in history_rows:
        search_id = history_row["SEARCH_ID"]

        if search_id not in master_by_search_id:
            raise ValueError(
                "Inquiry history has an unmatched SEARCH_ID: "
                f"{search_id}"
            )

        parent_row = master_by_search_id[search_id]

        if history_row["PIN_CODE"] != parent_row["PIN_CODE"]:
            raise ValueError(
                f"PIN_CODE mismatch for SEARCH_ID: {search_id}"
            )

        if (
            history_row["DOCUMENTNO"]
            != parent_row["DOCUMENT_NO"]
        ):
            raise ValueError(
                f"DOCUMENTNO mismatch for SEARCH_ID: {search_id}"
            )

        inquiry_date = parse_date(
            history_row["INQDATE"]
        )

        reporting_date = parse_date(
            history_row["REPORTINGDATE"]
        )

        if inquiry_date >= reporting_date:
            raise ValueError(
                "History inquiry is not earlier than the "
                f"current search. SEARCH_ID: {search_id}"
            )

        if (
            reporting_date - inquiry_date
        ).days > LOOKBACK_DAYS:
            raise ValueError(
                "History inquiry is older than two years. "
                f"SEARCH_ID: {search_id}"
            )

        rows_per_search[search_id] += 1

    return rows_per_search


# ============================================================
# FILE WRITING
# ============================================================

def write_csv(file_path, rows):
    """Writes inquiry history rows to CSV."""

    with open(
        file_path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=MKR_INQUIRY_HISTORY_COLUMNS
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# MAIN
# ============================================================

def main_generate_mkr_inquiry_history():
    customer_creation_dates = load_customers()
    master_rows = load_mkr_master()

    events_by_owner = build_customer_inquiry_events(
        master_rows=master_rows,
        customer_creation_dates=customer_creation_dates
    )

    history_rows = create_inquiry_history_rows(
        master_rows=master_rows,
        events_by_owner=events_by_owner
    )

    rows_per_search = validate_inquiry_history(
        master_rows=master_rows,
        history_rows=history_rows
    )

    full_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_inquiry_history_base.csv"
    )

    sample_output_file = (
        SAMPLE_DATA_FOLDER
        / "rs_mkr_inquiry_history_base_sample.csv"
    )

    summary_output_file = (
        RAW_DATA_FOLDER
        / "rs_mkr_inquiry_history_generation_summary.json"
    )

    write_csv(full_output_file, history_rows)
    write_csv(sample_output_file, history_rows[:100])

    searches_with_history = len(rows_per_search)
    searches_without_history = (
        len(master_rows) - searches_with_history
    )

    max_history_rows = max(
        rows_per_search.values(),
        default=0
    )

    summary = {
        "table_name": "RS_MKR_INQUIRY_HISTORY",
        "generated_rows": len(history_rows),
        "unique_parent_search_ids": len(master_rows),
        "parent_search_ids_with_history": searches_with_history,
        "parent_search_ids_without_history": (
            searches_without_history
        ),
        "maximum_history_rows_for_one_search": (
            max_history_rows
        ),
        "join_rule": (
            "RS_MKR_INQUIRY_HISTORY.SEARCH_ID "
            "= RS_MKR_MASTER.SEARCH_ID"
        ),
        "lookback_rule": (
            "INQDATE is before REPORTINGDATE and no more "
            "than 730 days earlier"
        ),
        "individual_customer_rule": (
            "PIN_CODE matches the parent MKR master PIN_CODE"
        ),
        "company_customer_rule": (
            "DOCUMENTNO matches the parent MKR master DOCUMENT_NO"
        ),
        "random_seed": RANDOM_SEED
    }

    with open(
        summary_output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(summary, file, indent=4)

    print("MKR inquiry history generation completed successfully.")
    print(f"Generated inquiry-history rows: {len(history_rows):,}")
    print(
        "Master searches with prior inquiries: "
        f"{searches_with_history:,}"
    )
    print(
        "Master searches with no prior inquiries: "
        f"{searches_without_history:,}"
    )
    print(
        "Maximum inquiry-history rows for one search: "
        f"{max_history_rows:,}"
    )
    print("Unmatched SEARCH_ID values: 0")
    print("Two-year lookback validation: passed")
    print(f"Full file: {full_output_file}")
    print(f"Sample file: {sample_output_file}")


main_generate_mkr_inquiry_history()